# LangChain Streaming

<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>
<b>About</b><br><br>

This notebook is derived from the following notebook, with modifications and extensions: https://github.com/AI-Engineering-bootcamp/ai-eng-nbs-public/blob/master/langchain-streaming-202503.ipynb
</div>

## Intro


Streaming means returning an LLM's output incrementally — token by token (or chunk by chunk) — as it's generated, instead of waiting for the entire response to finish before showing anything to the user.

**Why it matters**:
- **Perceived latency**: LLMs can take several seconds (or longer) to generate a full response. Streaming lets users see the first words almost immediately, making the application feel far more responsive even though the total generation time is the same.
- **Better UX for long outputs**: For long-form answers, reports, or code generation, users can start reading and reacting before the response is complete, rather than staring at a blank screen or loading spinner.
- **Early cancellation**: If a response is clearly going off-track, a user (or system) can stop generation early, saving time and cost.
- **Real-time interactivity**: Streaming is essential for chat-like interfaces (e.g., ChatGPT-style UIs), voice assistants, and any application where responsiveness is part of the product experience.

**Where it gets harder:**

Streaming a plain LLM call is straightforward — you just forward tokens as they arrive. But it becomes more complex once you introduce Agents, which run their own internal logic (tool calls, reasoning steps, intermediate "thoughts") before producing a final answer. Naively streaming an agent's raw output exposes all of that internal machinery (JSON actions, tool calls, etc.) instead of just the final, user-facing answer — so extra work is needed to filter and expose only what's relevant.

We'll start simple, streaming LLM output straight to the terminal, then work up to streaming an Agent's final answer while hiding its internal tool calls.


<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

In [2]:
# !pip install dotenv

<br>

## Initial Setup



In [3]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


<br>

## LLM Streaming to Stdout

The simplest form of streaming is to print each token as it's generated. To do this, we initialize an LLM (one that supports streaming — not all do) with two parameters:

* `streaming=True` — enables streaming.
* `callbacks=[SomeCallbackHere()]` — a LangChain callback (or list of callbacks) that runs code every time a new token arrives.

Here we use LangChain's built-in `StreamingStdOutCallbackHandler`, which simply prints each token as it's generated.



In [4]:
import os
from langchain_openai import ChatOpenAI
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

llm = ChatOpenAI(
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    model='gpt-6-luna',
    temperature=1,  # gpt-6-luna only accepts the default temperature
    max_completion_tokens=1000,
    streaming=True,  # ! important
    callbacks=[StreamingStdOutCallbackHandler()],  # ! important
)


# LangChain Agents pass a `stop` sequence (e.g. "\nObservation:"). 
# The model that we're using rejects that parameter, 
# so we're adding the custom function below to drop it before the request is sent.
def _payload(self, input_, *, stop=None, **kwargs):
    return _payload.original(self, input_, **kwargs)

_payload.original = getattr(ChatOpenAI._get_request_payload, "original", ChatOpenAI._get_request_payload)
ChatOpenAI._get_request_payload = _payload

/Users/luis/Desktop/ironhack_sept26/demos_ai_engineering_sept2026/.venv/langchain-v0.2/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: UserWarning: WARNING! max_completion_tokens is not default parameter.
                max_completion_tokens was transferred to model_kwargs.
                Please confirm that max_completion_tokens is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


**How it works**

1. `streaming=True` tells OpenAI: **"Don't wait until you've finished the entire answer—send it to me one small piece (token) at a time."**
2. Each time a new token arrives, LangChain immediately passes it to the callback we provided (in this case, `StreamingStdOutCallbackHandler`).

3. `StreamingStdOutCallbackHandler` simply prints each token to the terminal as soon as it receives it.


<br>

Now if we call the LLM, we'll see the response being _streamed_ token by token in the output below, instead of appearing all at once.

In [5]:
llm.invoke("Tell a long story where Willy Wonka invites the world's best AI engineers")

# Willy Wonka and the Thinking Machine

On a Monday morning, the world’s best AI engineers received identical gold envelopes.

They arrived by courier, by drone, and—in one case—inside a hollow chocolate teapot that had been delivered to the wrong office and politely returned three times before reaching its destination. Each envelope bore a name written in looping purple ink and sealed with a tiny, edible wax stamp.

Inside was a card:

> **Dear Engineer,**  
> You are invited to the Wonka Chocolate Factory for a week of invention, exploration, and a small amount of danger.  
> Bring your finest ideas. Leave your assumptions at the gate.  
> **Yours deliciously,**  
> **Willy Wonka**

At the bottom, in much smaller handwriting, someone had added:

> *P.S. Do not bring a robot that can open doors. The last one did.*

The engineers came from every corner of the world. They arrived with laptops, notebooks, toolkits, and enough charging cables to lasso a small moon. Some came by train. Som

AIMessage(content='# Willy Wonka and the Thinking Machine\n\nOn a Monday morning, the world’s best AI engineers received identical gold envelopes.\n\nThey arrived by courier, by drone, and—in one case—inside a hollow chocolate teapot that had been delivered to the wrong office and politely returned three times before reaching its destination. Each envelope bore a name written in looping purple ink and sealed with a tiny, edible wax stamp.\n\nInside was a card:\n\n> **Dear Engineer,**  \n> You are invited to the Wonka Chocolate Factory for a week of invention, exploration, and a small amount of danger.  \n> Bring your finest ideas. Leave your assumptions at the gate.  \n> **Yours deliciously,**  \n> **Willy Wonka**\n\nAt the bottom, in much smaller handwriting, someone had added:\n\n> *P.S. Do not bring a robot that can open doors. The last one did.*\n\nThe engineers came from every corner of the world. They arrived with laptops, notebooks, toolkits, and enough charging cables to lasso 

<br>

That was surprisingly easy — but streaming gets more complicated once we introduce agents, which reason over multiple steps and call tools before producing a final answer. Let's initialize an agent to see why.

In [6]:
from langchain.memory import ConversationBufferWindowMemory
from langchain.agents import AgentType, initialize_agent
from langchain.tools import Tool

# initialize conversational memory
memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    k=5,
    return_messages=True,
    output_key="output"
)

# create tools for the agent
def lookup_shipping_days(country: str) -> str:
    """Returns an estimated shipping time for a destination country."""
    shipping = {
        "Spain": "2–3 business days",
        "France": "3–4 business days",
        "Japan": "5–8 business days",
    }
    return shipping.get(country.strip().title(), "Shipping estimate unavailable.")

tools = [
    Tool(
        name="lookup_shipping_days",
        func=lookup_shipping_days,
        description="Look up the estimated shipping time in business days for a destination country. Input a country name, such as Spain, France, or Japan.",
    )
]

# initialize the agent
agent = initialize_agent(
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    tools=tools,
    llm=llm,
    memory=memory,
    verbose=False, # set to True for detailed logging of agent steps
    max_iterations=3,
    early_stopping_method="generate",
    return_intermediate_steps=False
)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_79091/907337456.py:32: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


<br>

The agent reuses the same `llm` object, so it already has our `StreamingStdOutCallbackHandler` attached. Let's see what streaming looks like when running the agent.

In [7]:
prompt = "Hello, how are you?"

result_1 = agent(prompt)

# print(result_1) # we can keep this commented ('StreamingStdOutCallbackHandler' will automatically print each token in the terminal)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_79091/150748627.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  result_1 = agent(prompt)


```json
{
  "action": "Final Answer",
  "action_input": "Hello! I'm doing well, thanks for asking. How are you?"
}
```

<br>

Not bad, but now we're streaming the LLM's _entire_ raw output — including the JSON structure the agent uses internally to decide on tool calls and reasoning steps. This is useful for debugging, but not something we'd want to show end users. 

Let's see another example where we ask a shipping question that requires a tool call:

In [8]:
query_2 = """
    I want to purchase a product, how long will it take to ship to Spain?

    Also, after answering that question, add please two line breaks, 
    and 5 paragraphs explaining the process involved to ship a product to Spain.
    """

# note: we're asking the LLM to include an extra paragraph, 
# so that we can see the streaming in action.


result_2 = agent(query_2)

# print(result_2)

```json
{
  "action": "lookup_shipping_days",
  "action_input": "Spain"
}
``````json
{
  "action": "Final Answer",
  "action_input": "The estimated shipping time to Spain is 2–3 business days, based on the estimate provided.\n\nAfter you place your order, the seller confirms the purchase and checks that the product is available. You’ll typically receive an order confirmation with details about your purchase.\n\nThe product is then picked, packed, and labeled for shipment. The seller prepares the required shipping documents and arranges for a carrier to collect the parcel.\n\nOnce collected, the carrier sorts the parcel and transports it toward Spain. You may be able to follow its progress using a tracking number, if one is provided.\n\nIf the parcel is shipping internationally, it may need to pass through customs. Any customs checks or required charges can depend on the product and its country of origin.\n\nFor the final stage, a local delivery service brings the parcel to the address 

Looking at the streamed output above, we can see the raw JSON "thoughts" the agent streams token-by-token as it works through the problem.

This is exactly the problem we mentioned earlier: streaming shows us all of this internal JSON reasoning, when really we'd only want to stream the final answer text to an end user.

<br>

The raw output can be very useful in some cases (e.g. debugging), but in many other cases you only want to stream the final answer. 

We have two options: 
- Write a custom callback handler
- Use LangChain's purpose-built `FinalStreamingStdOutCallbackHandler`. 

<br>

Let's try the built-in one first. We'll replace the LLM's current callbacks (still our `StreamingStdOutCallbackHandler` from before):

In [9]:
from langchain.callbacks.streaming_stdout_final_only import FinalStreamingStdOutCallbackHandler

# Configure the LLM to begin streaming only after the model emits the
# tokens ["Final", "Answer"].
agent.agent.llm_chain.llm.callbacks = [
    FinalStreamingStdOutCallbackHandler(
        answer_prefix_tokens=["Final", "Answer"]
    )
]

Let's try it out:

In [10]:
result_3 = agent(query_2)

# print(result_3)

",
  "action_input": "Shipping to Spain is estimated to take 2–3 business days.\n\nAfter you place your order, the seller confirms it and prepares the product for dispatch.\n\nThe product is picked, packed, and labeled, and the seller arranges for a carrier to collect it.\n\nThe carrier transports the parcel toward Spain. If tracking is available, you can use the tracking number to follow its progress.\n\nFor international shipments, the parcel may be subject to customs checks or charges, depending on its origin and contents.\n\nA local carrier then completes delivery to your address. The estimate can vary with the shipment’s origin, carrier, customs processing, and other delays."
}
```

<br>

Still not quite clean — tuning `answer_prefix_tokens` correctly is finicky. It's usually easier to write a custom callback that filters tokens ourselves, only printing once it detects we're inside the `action_input` of the final answer:

In [11]:
import re
import sys
from langchain.callbacks.base import BaseCallbackHandler

class FinalAnswerStreamingCallbackHandler(BaseCallbackHandler):
    def __init__(self):
        self._reset()

    def _reset(self):
        self.content = ""
        self.final_answer = False
        self.in_action_input = False
        self.escaped = False
        self.unicode_escape = None
        self.answer_done = False

    def on_llm_start(self, serialized, prompts, **kwargs):
        self._reset()

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self._reset()

    def on_llm_new_token(self, token, **kwargs):
        if self.answer_done:
            return

        if not self.final_answer:
            self.content += token
            marker = "Final Answer"
            marker_index = self.content.find(marker)
            if marker_index == -1:
                self.content = self.content[-(len(marker) - 1):]
                return
            self.content = self.content[marker_index + len(marker):]
            self.final_answer = True
        else:
            self.content += token

        if not self.in_action_input:
            match = re.search(r'"action_input"\s*:\s*"', self.content)
            if match is None:
                return
            self.content = self.content[match.end():]
            self.in_action_input = True

        for character in self.content:
            if self.unicode_escape is not None:
                self.unicode_escape += character
                if len(self.unicode_escape) == 4:
                    sys.stdout.write(chr(int(self.unicode_escape, 16)))
                    self.unicode_escape = None
                    self.escaped = False
            elif self.escaped:
                if character == "u":
                    self.unicode_escape = ""
                    self.escaped = False
                else:
                    escaped_characters = {
                        '"': '"',
                        "\\": "\\",
                        "/": "/",
                        "b": "\b",
                        "f": "\f",
                        "n": "\n",
                        "r": "\r",
                        "t": "\t",
                    }
                    sys.stdout.write(escaped_characters.get(character, character))
                    self.escaped = False
            elif character == "\\":
                self.escaped = True
            elif character == '"':
                self.answer_done = True
                break
            else:
                sys.stdout.write(character)

        sys.stdout.flush()
        self.content = ""

callback_handler = FinalAnswerStreamingCallbackHandler()
agent.agent.llm_chain.llm.callbacks = [callback_handler]

<br>

Let's try again with our custom handler:

In [12]:
result_4 = agent(query_2)

# print(result_4)

The estimated shipping time to Spain is 2–3 business days.

After you place your order, the seller confirms the purchase and checks that the product is available.

The product is then picked, packed, and labeled for shipment, and the seller arranges for a carrier to collect it.

The carrier transports the parcel toward Spain. If tracking is provided, you can use the tracking number to follow its progress.

For international shipments, the parcel may be subject to customs checks or charges, depending on its origin and contents.

A local delivery carrier completes the final delivery to your address. Delivery times can vary due to the shipment’s origin, customs processing, or other delays.